In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
import requests
import urllib.request
import json

In [14]:
url = "https://query1.finance.yahoo.com/v8/finance/chart/TSLA?range=max&interval=1d"

req = urllib.request.Request(
    url,
    headers={"User-Agent": "Mozilla/5.0"}
)

with urllib.request.urlopen(req) as response:
    data = json.loads(response.read().decode())

result = data['chart']['result'][0]
timestamps = result['timestamp']
quotes = result['indicators']['quote'][0]

df = pd.DataFrame({
    "Date": pd.to_datetime(timestamps, unit='s'),
    "Close": quotes['close'],
    "High": quotes['high'],
    "Low": quotes['low'],
    "Open": quotes['open'],
    "Volume": quotes['volume']
})

df["Date"] = df["Date"].dt.strftime('%Y-%m-%d')
df = df.dropna().reset_index(drop=True)

required_columns = ["Date", "Open", "High", "Low", "Close", "Volume"]
missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

df = df[["Date", "Open", "High", "Low", "Close", "Volume"]].copy()
df.head()

,Date,Open,High,Low,Close,Volume
0,2010-07-01,1.666667,1.728000,0.998667,1.329333,968637000
1,2010-08-01,1.366667,1.478667,1.159333,1.298667,225573000
2,2010-09-01,1.308000,1.544000,1.300000,1.360667,270688500
3,2010-10-01,1.379333,1.458000,1.333333,1.456000,98217000
4,2010-11-01,1.462667,2.400000,1.403333,2.355333,424726500


In [15]:
forecast_horizon = 4

df["future_close"] = df["Close"].shift(-forecast_horizon)
df["target"] = (df["future_close"] > df["Close"] * 1.01).astype(int)

df[["Date", "Close", "future_close", "target"]].head(10)

# Returns
df["return_1"] = df["Close"].pct_change(1)
# 1-day return: captures immediate price movement between two consecutive periods

df["return_3"] = df["Close"].pct_change(3)
# 3-day return: short-term trend indicator

df["return_5"] = df["Close"].pct_change(5)
# 5-day return: weekly momentum proxy

df["return_10"] = df["Close"].pct_change(10)
# 10-day return: medium-term momentum

# Moving Means
df["ma_5"] = df["Close"].rolling(5).mean()
# 5-period moving average: smooths short-term noise

df["ma_10"] = df["Close"].rolling(10).mean()
# 10-period moving average: short-term trend indicator

df["ma_20"] = df["Close"].rolling(20).mean()
# 20-period moving average: common baseline trend level

# Moving means ratio
df["ma_ratio_5"] = df["Close"] / df["ma_5"]
# Distance from short-term trend: detects overbought / oversold conditions

df["ma_ratio_10"] = df["Close"] / df["ma_10"]
# Measures deviation from 10-period trend

df["ma_ratio_20"] = df["Close"] / df["ma_20"]
# Measures deviation from medium-term trend

# Volatility
df["volatility_5"] = df["Close"].pct_change().rolling(5).std()
# Short-term realized volatility (5 periods)

df["volatility_10"] = df["Close"].pct_change().rolling(10).std()
# Medium-term realized volatility

# Momentum
df["momentum_5"] = df["Close"] - df["Close"].shift(5)
# Absolute price momentum over 5 periods

df["momentum_10"] = df["Close"] - df["Close"].shift(10)
# Medium-term momentum

# Volume
df["volume_change_1"] = df["Volume"].pct_change(1)
# Detects sudden volume spikes or drops

df["volume_change_5"] = df["Volume"].pct_change(5)
# Measures sustained volume trend

# Daily Range
df["daily_range"] = (df["High"] - df["Low"]) / df["Close"]
# Intraday volatility: captures market activity and price spread

# Open-close Shift
df["open_close_diff"] = (df["Close"] - df["Open"]) / df["Open"]
# Measures intraday bullish/bearish pressure

delta = df["Close"].diff()
# Price change between two consecutive periods

gain = delta.clip(lower=0)
# Keeps only positive price movements (up moves)

loss = -delta.clip(upper=0)
# Keeps only negative price movements (down moves)

avg_gain = gain.rolling(14).mean()
# Average gain over the last 14 periods

avg_loss = loss.rolling(14).mean()
# Average loss over the last 14 periods

rs = avg_gain / avg_loss
# Relative strength: ratio of buying pressure vs selling pressure

df["RSI"] = 100 - (100 / (1 + rs))
# Relative Strength Index (0–100): momentum oscillator used to detect overbought (>70) or oversold (<30) conditions

bb_ma20 = df["Close"].rolling(20).mean()
# 20-period moving average used as the central trend for Bollinger Bands

bb_std20 = df["Close"].rolling(20).std()
# 20-period standard deviation measuring price dispersion around the mean

bollinger_upper = bb_ma20 + 2 * bb_std20
# Upper Bollinger Band: dynamic resistance level based on volatility

bollinger_lower = bb_ma20 - 2 * bb_std20
# Lower Bollinger Band: dynamic support level based on volatility

df["bollinger_position"] = (
    (df["Close"] - bollinger_lower) / (bollinger_upper - bollinger_lower)
)
# Normalized position of the price within the Bollinger Bands (0 = lower band, 1 = upper band)
# Helps detect overbought (>1) or oversold (<0) conditions and volatility expansions

ema12 = df["Close"].ewm(span=12, adjust=False).mean()
# Fast exponential moving average capturing short-term price trend

ema26 = df["Close"].ewm(span=26, adjust=False).mean()
# Slow exponential moving average capturing medium-term trend

df["MACD"] = ema12 - ema26
# Moving Average Convergence Divergence: measures momentum by comparing short vs medium trends

df["MACD_signal"] = df["MACD"].ewm(span=9, adjust=False).mean()
# Smoothed signal line used to detect momentum shifts

df["MACD_diff"] = df["MACD"] - df["MACD_signal"]
# MACD histogram: measures the strength and direction of momentum changes

df["trend_strength_20"] = df["Close"] / df["ma_20"]
# Measures how far the price is from the 20-period trend (trend strength indicator)

df["above_ma20"] = (df["Close"] > df["ma_20"]).astype(int)
# Binary indicator: 1 if price is above the 20-period trend, 0 otherwise

df["acceleration"] = df["return_1"].diff()
# Captures change in momentum (second derivative of price movement)

df["trend_slope"] = df["ma_20"].diff()
# Measures the slope of the 20-period moving average (trend direction)

df["distance_ma20"] = (df["Close"] - df["ma_20"]) / df["ma_20"]
# Normalized distance from the 20-period moving average (trend deviation)

df["trend_slope_50"] = df["Close"].rolling(50).mean().diff()
# Slope of the long-term trend (50-period moving average)

df["distance_ma50"] = (df["Close"] - df["Close"].rolling(50).mean()) / df["Close"].rolling(50).mean()
# Distance from long-term trend level

df["volatility_50"] = df["Close"].pct_change().rolling(50).std()
# Long-term realized volatility

df["vol_regime"] = df["volatility_10"] / df["volatility_50"]
# Detects volatility regime changes (short vs long volatility)

df["volume_ma20"] = df["Volume"].rolling(20).mean()
# Smoothed average trading volume

df["volume_shock"] = df["Volume"] / df["volume_ma20"]
# Detects abnormal volume spikes relative to recent average

for lag in range(1,6):
    df[f"return_lag_{lag}"] = df["return_1"].shift(lag)
# Adds past returns as lag features to capture short-term temporal patterns

features = [
    "distance_ma20",
    "volatility_50",
    "distance_ma50",
    "trend_strength_20",
    "ma_ratio_20",
    "MACD_signal",
    "MACD",
    "trend_slope",
    "RSI",
    "volatility_10",
    "ma_ratio_10",
    "MACD_diff",
    "return_10",
    "trend_slope_50",
    "vol_regime"
]

df[features] = df[features].shift(1)
# Shifts all feature values by one period to ensure the model only uses past information
# This prevents data leakage by avoiding the use of current or future data when predicting the next movement

df = df.dropna().reset_index(drop=True)
# Removes rows with missing values created by rolling indicators and feature shifts
# Ensures the dataset only contains valid samples for model training

print(df.shape)
# Displays the dataset size after cleaning (number of rows and features)

df[["Date", "Close", "target"] + features[:5]].head()
# Quick inspection of the dataset: shows date, price, target and a few engineered features
# Useful to verify that feature engineering and shifting were applied correctly

X = df[features]
y = df["target"]

split_index = int(len(df) * 0.8)
# Defines the split point using 80% of the dataset for training

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]
# Time-based split: first part for training, later data for testing

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]
# Targets are split using the same temporal boundary

print("Train size:", len(X_train))
print("Test size:", len(X_test))
# Displays the number of samples used for training and evaluation

boost_classifier_model_1 = XGBClassifier(
    n_estimators=1200,
    # Number of boosting trees used to build the model (more trees → more capacity)

    max_depth=5,
    # Maximum depth of each tree controlling model complexity

    learning_rate=0.01,
    # Step size used to update model weights (lower values improve generalization)

    subsample=0.9,
    # Fraction of training samples used for each tree (reduces overfitting)

    colsample_bytree=0.9,
    # Fraction of features randomly used when building each tree

    gamma=1,
    # Minimum loss reduction required to create a new split (regularization)

    reg_alpha=0.1,
    # L1 regularization term helping to reduce overfitting

    reg_lambda=1,
    # L2 regularization term stabilizing model weights

    random_state=42,
    # Ensures reproducibility of the training process

    n_jobs=-1
    # Uses all CPU cores to speed up training
)

boost_classifier_model_1.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    # Uses the test set as evaluation dataset during training

    verbose=False
    # Disables training logs output
)

y_pred = boost_classifier_model_1.predict(X_test)

y_proba = boost_classifier_model_1.predict_proba(X_test)

# Probabilité associée à la classe prédite
prediction_confidence = y_proba.max(axis=1)

# En pourcentage
prediction_confidence_percent = prediction_confidence * 100

# Dataset de comparaison sur le dernier mois
df_compare = df.iloc[split_index:].copy()

df_compare["prediction"] = y_pred
df_compare["actual"] = y_test.values

# Erreur ou réussite
df_compare["is_correct"] = df_compare["prediction"] == df_compare["actual"]

# Label lisible
df_compare["result"] = df_compare["is_correct"].map({
    True: "correct",
    False: "wrong"
})

df_compare["prediction_confidence"] = prediction_confidence_percent

# Garder uniquement le dernier mois
last_date = df_compare["Date"].max()
one_month_ago = last_date - pd.DateOffset(months=1)

df_last_month = df_compare[df_compare["Date"] >= one_month_ago].copy()

# Colonnes utiles
df_last_month_result = df_last_month[
    [
        "Date",
        "Close",
        "future_close",
        "target",
        "prediction",
        "prediction_confidence",
        "result"
    ]
]

df_last_month_result

df_last_month_result["target_label"] = df_last_month_result["target"].map({

    1: "BTC +1% dans 5 jours",

    0: "Pas de hausse +1%"

})

df_last_month_result["prediction_label"] = df_last_month_result["prediction"].map({

    1: "Modèle prédit +1%",

    0: "Modèle prédit pas +1%"

})

df_last_month_result

df_last_month_result["target_label"] = df_last_month_result["target"].map({
    1: "BTC +1% dans 5 jours",
    0: "Pas de hausse +1%"
})

df_last_month_result["prediction_label"] = df_last_month_result["prediction"].map({
    1: "Modèle prédit +1%",
    0: "Modèle prédit pas +1%"
})

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

importances = pd.Series(boost_classifier_model_1.feature_importances_, index=features)
importances.sort_values(ascending=False).head()

(138, 47)
Train size: 110
Test size: 28
Accuracy: 0.6785714285714286


MACD              0.104116
volatility_10     0.092861
return_10         0.089278
trend_slope_50    0.088076
ma_ratio_10       0.078349
dtype: float32